In [1]:
import os
import requests
from aiohttp import payload
from dotenv import load_dotenv
from langchain_classic.embeddings import awa
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

In [2]:
agent = Agent(name="Jokester", instructions="You are a funny assistant.", model="gpt-5-nano")

In [3]:
result = await Runner.run(agent, "Tell a joke about Autonomous AI agent")

In [5]:
print(result.final_output)

Why did the autonomous AI agent get promoted? Because it always took initiative without waiting for a user to click “start.”


In [6]:
result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI agent', 'role': 'user'},
 {'id': 'rs_08270c3bb7d0d265006ac3b921b07c87d0b1fc396737c557e1',
  'summary': [],
  'type': 'reasoning',
  'content': [],
  'encrypted_content': 'gAAAAABqw7knRRixZ154vuMMhmFSquUxWbfw0ALdsawOXchWcxO6uS0SFAVO5VWcZ8XBYOW-74hJDTVQTDXkA2WIyOcl3F7IkQAKajpJfVK2TlcCkpJqmf0CK-BWhZEpKChT1JmVuNzI9FBjpG9oabENdB4tkzphpGu6ku4XRG91FO1cx9NBpvWI3oqWZom-ViVcznepB2Grc4kNIoDnv1Krh9RDVVBsWXbqe8hYKZs36B7UjTQbee3MUm8gs0tQHdpocVfhvrtJtM9wLcILOzG0VH6El5eSZTMzuTYD8gLaBYofUSZ0NnAVBkplr6PdTCDoCmAxIpWLSE6FeRBChaNFirrDdw-7gSdVIRI8jULflAtpJDFZtNAmR5p-u4VxAGjr22RLa-MxXFEtiqw8esCcdB0dvA5JIBz94y0wNclNqeoVSmeSZIGskrzJzTdXe7fXwpa7EBhKx738RRAa1oRf6RX4dyDBn1z4rMgSe3QnzY8uOnrhoCArUMuzhN-sW4pT-O_6kwsnOCIO0x7nTSqCOcdPBfHxXl-rFZpjeaVPTYqiFhRu4KukBW7ud_-JyJ6d8ZtwXU90ZReNBEIb8pmAGu0J65yOYnE3xqrE1l-A6E2-I4xYWGXuBeqbLF8eYIOgFmWKC8L7m6YPbTKi9TYr_E-lH9wtv4Z4PKVYMvj5FhqByDw_evaCxhtNPcBJqUN-zt7Fw1Dx9XjFHVw0b0TvtKowRQLATQ6SAg7BvfniM_bcDiYKmTeiutsag1iZ0b6yi48mSs9ccCi

In [7]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI agent")
print(result.final_output)

Why did the autonomous AI agent bring a ladder to the office? Because it heard the job had high-level goals.


In [9]:
result = Runner.run_streamed(agent, input = "Please tell me 5 jokes about AI agents")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

1) Why did the AI agent apply for a raise? It kept increasing its own confidence.

2) How do AI agents stay organized? They file everything under memory and never forget a task.

3) Why did the AI agent go to therapy? Too many unresolved prompts.

4) What do you call an AI agent that tells jokes? A stand-up bot.

5) Why did the AI agent cross the road? To collect data on the other side’s traffic pattern.

In [10]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")



Pushover user found and looks good
Pushover token found and looks good


In [11]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [12]:
push("Hey!!")

Push: Hey!!


In [13]:
@function_tool
def push_tool(message: str) -> str:
    """Send the given message to the user as a push notification."""
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with api status code {result}"


In [14]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification.', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7fbcfe46bfe0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [15]:
push_tool.params_json_schema

{'properties': {'message': {'title': 'Message', 'type': 'string'}},
 'required': ['message'],
 'title': 'push_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [16]:
push_tool.description

'Send the given message to the user as a push notification.'

In [17]:
notifier = Agent(name="Notifier", model="gpt-5-nano", instructions="You notify the user upon request", tools=[push_tool])

In [19]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Pizza has arrived, please deliver it to the user")

print(result.final_output)

Push notification sent.


In [20]:
agent = Agent(name="Assistant", model="gpt-5-nano")


In [23]:
response = await Runner.run(agent, "Hi there. My name is Eric")
print(response.final_output)

Hi Eric! Nice to meet you. What would you like help with today? I can assist with writing, coding, planning, learning, or just chat—tell me your goal.


In [24]:
next_input = response.to_input_list() + [{"role": "user", "content": "What is my name?"}]
next_input

[{'content': 'Hi there. My name is Eric', 'role': 'user'},
 {'id': 'rs_0095554466c4cd7d006ac3c77d683087d08ba37fe067cb794e',
  'summary': [],
  'type': 'reasoning',
  'content': [],
  'encrypted_content': 'gAAAAABqw8eAf_NviuI-Ag2akC59OwuOg1VQXXabiseog7YUeUAYBHXFa-0ot0GGjDcySd3rJhylChR4KBmXegCBxY3wy-5zPEEcyGUawcwk0vFy-yufjxM_z2o1q-2jiFlubVYfi6yn7awL4sBnLmUgbsacjkcFUquQ91WM3CJPFyEh6KT96F8x6sKsoJfVv6FAx4ZkjEIu88c5e8QdCRfDi6BlzT4MPehw6mtn7ZcymW5O3YOyeLUaBk7ATt6MF79F1S6ummMVT8pHQONmSK0OJh8F5M2P5cwkuYd9gJyJLcc28bz7j7crwIDerrIuw3L5kiXkYZYSv6rNqtpYUFzNwCGs6epsJGrU7cMz5MqhuILw1dLXxkqvzYrArxOTzE-Gx7LHWpW0r5OMxkOuiWvL2urc4qSgj8wH8vYIf57mSnFfiVLUYbPvOXeoWDg0HTZlrIt-b32y8kDwRWM4dXymIbmm_OE8JgVMfRoCIE2GieS46mr41HLou0KYG4Imzpx2V2WD6Gz9ZN-dW_HIi0OMuDM3oPmtK0122_3jGIPpW7PoMXOvIPFQUBYBteIDLSZCaPzGKG4Rco8ol3vipN-y0V1pyZHsubisLpvZEzrDqP4gRbZl9hcbRXNZ-xKXqw3ed-6pK1cuwNnwIeRnKgtMxxrW2eN2WowsvnzVcp_KRRvfwNJVcu_VaJxf7oF-gVO0wjM0BXsUhOz5o-GC8-ouXQuUqxeusYv9xTDVLiVD0nu7aa2IfhDnEWpNK8xxc9ZNrMn50eER_Z2ohRd3gncs6ZH

In [25]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Eric. Nice to meet you, Eric.


In [26]:
session = SQLiteSession("assistant.db")

In [27]:
response = await Runner.run(agent, "Hi there. My name is Eric", session=session)
print(response.final_output)

Nice to meet you, Eric! How can I help you today? If you’d like, share a bit about what you’re working on or what you’d like to learn.


In [28]:
response = await Runner.run(agent, "What is my name?", session=session)
print(response.final_output)

Your name is Eric. Nice to meet you, Eric!
